In [1]:
import src.database.scripts.sql as sql
from src.database.data_collectors.item_data_fetch import item_data_fetch

In [2]:
def drop_item_data_table(cursor):
    query = "DROP TABLE IF EXISTS item_data"
    cursor.execute(query)

def create_item_data_table(cursor):
    columns = body_list[0].keys()
    type_map = {}
    for col in columns:
        for row in body_list:
            col_type = type(row[col])
            if col_type != type(None):
                if col_type == int:
                    type_map[col] = "INT"
                if col_type == float:
                    type_map[col] = "FLOAT"            
                if col_type == str:
                    type_map[col] = "TEXT"
                break
        else:
            type_map[col] = "TEXT"
    query_col_text = ", ".join(f"{col} {dtype}" for col, dtype in type_map.items())
    
    query = f"""
        CREATE TABLE IF NOT EXISTS item_data ({query_col_text}
        )"""
    cursor.execute(query)
    
    query = f"""
        ALTER TABLE item_data
        ADD CONSTRAINT unique_item_id UNIQUE (id);
        """
    cursor.execute(query)

def insert_item_data():
    columns = body_list[0].keys(cursor)
    rows = [tuple(row[col] for col in columns) for row in body_list]

    query_col_text = ", ".join(columns)
    row_placeholder = ", ".join(["%s"] * len(columns))
    
    query = f"""
        INSERT INTO item_data ({query_col_text})
        VALUES ({row_placeholder})
        """
    cursor.executemany(query, rows)

def vendor_price_correction(cursor):
    query = """
        UPDATE item_data
        SET vendor_price = CASE id
            WHEN 'CrackedLog' THEN 75
            WHEN 'SturdyLog'  THEN 150
        END
        WHERE id IN ('CrackedLog', 'SturdyLog');
        """
    cursor.execute(query)

def update_item_data(cursor):
    body_list = item_data_fetch()
    columns = body_list[0].keys()
    rows = [tuple(row[col] for col in columns) for row in body_list]

    query_col_text = ", ".join(columns)
    row_placeholder = ", ".join(["%s"] * len(columns))
    set_clause = ", ".join(f"{col} = EXCLUDED.{col}" for col in columns if col != 'id')

    query = f"""
        INSERT INTO item_data ({query_col_text})
        VALUES ({row_placeholder})
        ON CONFLICT (id) DO UPDATE
        SET {set_clause}
        """
    cursor.executemany(query, rows)

In [3]:
def main():
    conn = sql.connect_pc()
    cursor = conn.cursor()

    print("dropping existing table...")
    drop_item_data_table(cursor)

    print("fetching data...")
    body_list = item_data_fetch(cursor)

    print("creating table...")
    create_item_data_table(cursor)

    print("inserting data...")
    insert_item_data(cursor)

    print("correcting vendor price")
    vendor_price_correction(cursor)

    conn.commit()
    conn.close()
    print("compelete!")

def update():
    conn = sql.connect_pc()
    cursor = conn.cursor()

    update_item_data(cursor)
    vendor_price_correction(cursor)

    conn.commit()
    conn.close()

In [4]:
update()